# Обучение агента в Unity ML-Agents с экспортом ONNX

Этот ноутбук обучает агента из Unity-среды (`MyAgent?team=0`) с 6 наблюдениями и 2 непрерывными действиями (без прыжка). Используем PPO из `stable-baselines3` с поддержкой GPU и экспортируем модель в ONNX для Unity Sentis.

## Зависимости
- Установите библиотеки:
  ```bash
  pip install mlagents==0.30.0 stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html onnx numpy psutil
  ```
- Убедитесь, что путь к `UnityEnvironment.exe` правильный.
- Среда Unity должна быть собрана с `Behavior Name: MyAgent?team=0`, `Continuous Actions: 2`, `Discrete Actions: 0`.
- Python 3.8, NVIDIA GPU с CUDA 11.8 (или совместимая версия).
- Проверьте доступность GPU:
  ```bash
  python -c "import torch; print(torch.cuda.is_available())"
  ```

Conda python setup
Once conda has been installed in your system, open a terminal and execute the following commands to setup a python 3.8.20 virtual environment and activate it.
# conda create -n mlagents python=3.8.20 && conda activate mlagents


In [ ]:
# Эти строки закомментированы, так как мы используем предустановленные библиотеки
# Раскомментируйте эти строки, если вы работаете с исходным кодом ml-agents

# !pip3 install torch -f https://download.pytorch.org/whl/torch_stable.html
# Установить PyTorch с поддержкой CUDA (для ускорения на GPU)

# !pip3 install -e ./ml-agents-envs
# Установить библиотеку ML-Agents в режиме разработки (из локальной папки)

# !pip3 install -e ./ml-agents
# Установить основную библиотеку ML-Agents в режиме разработки

In [4]:
# Установка специфичной версии mlagents для совместимости
!pip3 install mlagents 

# Установка необходимых библиотек для обучения агента с поддержкой GPU
# !pip3 install stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html

  Using cached grpcio-1.48.2-cp310-cp310-win_amd64.whl.metadata (4.0 kB)
  Using cached numpy-1.23.5-cp310-cp310-win_amd64.whl.metadata (2.3 kB)
Using cached grpcio-1.48.2-cp310-cp310-win_amd64.whl (3.6 MB)
Using cached numpy-1.23.5-cp310-cp310-win_amd64.whl (14.6 MB)
   ---------------------------------------- 0.0/111.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/111.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/111.0 MB 3.4 MB/s eta 0:00:33
    --------------------------------------- 2.1/111.0 MB 4.1 MB/s eta 0:00:27
    --------------------------------------- 2.6/111.0 MB 4.1 MB/s eta 0:00:27
   - -------------------------------------- 3.7/111.0 MB 4.0 MB/s eta 0:00:28
   - -------------------------------------- 4.2/111.0 MB 3.8 MB/s eta 0:00:29
   - -------------------------------------- 5.0/111.0 MB 3.9 MB/s eta 0:00:28
   - -------------------------------------- 5.5/111.0 MB 3.8 MB/s eta 0:00:28
   -- -------------------------------------

  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
manim 0.19.0 requires numpy>=2.1; python_version >= "3.10", but you have numpy 1.23.5 which is incompatible.
torchaudio 2.7.1+cu118 requires torch==2.7.1+cu118, but you have torch 2.9.1 which is incompatible.
torchvision 0.15.2+cu118 requires torch==2.0.1, but you have torch 2.9.1 which is incompatible.


In [5]:
# Импорт необходимых библиотек
import os  # Для работы с путями файлов и переменными окружения
import numpy as np  # Библиотека для работы с массивами и математическими операциями
import torch  # PyTorch - библиотека для глубокого обучения с поддержкой GPU
import torch.nn as nn  # Модуль для создания нейронных сетей в PyTorch

# Импорт компонентов ML-Agents для работы с Unity-средой
from mlagents_envs.environment import UnityEnvironment  # Класс для подключения к Unity-среде
from mlagents_envs.side_channel.engine_configuration_channel import EngineConfigurationChannel  # Канал для настройки параметров симуляции Unity
from mlagents_envs.base_env import ActionTuple  # Класс для передачи действий агенту

# Импорт компонентов Stable-Baselines3 для обучения с подкреплением
from stable_baselines3 import PPO  # Алгоритм PPO (Proximal Policy Optimization) для обучения
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor  # Базовый класс для создания пользовательских нейросетей
from stable_baselines3.common.env_checker import check_env  # Функция для проверки совместимости среды

# Импорт компонентов Gymnasium (стандартное API для создания сред)
import gymnasium as gym  # Фреймворк для создания игровых окружений
from gymnasium import spaces  # Классы для определения пространств наблюдений и действий

# Определение функции для безопасного закрытия Unity-среды
def close_unity_env(env):  # Функция, которая закрывает среду и останавливает процессы
    try:  # Блок try - попытка выполнить код
        if env is not None:  # Проверка, если среда инициализирована (не пуста)
            env.close()  # Закрыть среду Unity
            print('Среда Unity успешно закрыта.')  # Вывод сообщения об успехе
        else:  # Если среда не инициализирована
            print('Среда не инициализирована.')  # Вывести сообщение
    except Exception as e:  # Блок except - обработка ошибок
        print(f'Ошибка при закрытии среды: {e}')  # Вывести информацию об ошибке
    finally:  # Блок finally - выполняется всегда, даже если была ошибка
        import psutil  # Импорт библиотеки для работы с процессами
        for proc in psutil.process_iter(['name']):  # Перебрать все запущенные процессы
            if proc.info['name'].lower() == 'unityenvironment.exe':  # Если процесс - UnityEnvironment
                proc.kill()  # Принудительно завершить процесс
                print(f'Процесс UnityEnvironment.exe (PID: {proc.pid}) принудительно завершён.')  # Вывести информацию
            if 'unity' in proc.info['name'].lower():  # Если в названии процесса есть слово "unity"
                proc.kill()  # Принудительно завершить процесс
                print(f"Процесс {proc.info['name']} (PID: {proc.pid}) принудительно завершён.")  # Вывести информацию

# Задание пути к исполняемому файлу Unity-среды
env_path = os.path.join(os.getcwd(), r'N:\MyRL\My_First_NPC\Environment\UnityEnvironment.exe')  # Объединить текущую директорию с путём к Unity-среде

# Создание канала для настройки параметров симуляции Unity
engine_channel = EngineConfigurationChannel()  # Инициализировать канал конфигурации
engine_channel.set_configuration_parameters(time_scale=20.0, quality_level=0)  # Установить ускорение симуляции в 20 раз и минимальное качество графики

# Инициализация среды Unity с обработкой ошибок
try:  # Блок try - попытка инициализировать среду
    env = UnityEnvironment(file_name=env_path, worker_id=1, base_port=6000, side_channels=[engine_channel], timeout_wait=60)  # Создать объект среды с параметрами
    env.reset()  # Выполнить первый сброс среды
except Exception as e:  # Если произойдёт ошибка
    print(f'Ошибка инициализации среды: {e}')  # Вывести сообщение об ошибке
    close_unity_env(None)  # Закрыть среду (if it was partially initialized)
    raise  # Повторно выбросить исключение

# Получение имени поведения агента (первое зарегистрированное поведение)
behavior_name = list(env.behavior_specs.keys())[0]  # Получить первое имя поведения из словаря
print(f'Behavior Name: {behavior_name}')  # Вывести имя поведения
spec = env.behavior_specs[behavior_name]  # Получить спецификацию поведения (описание входов/выходов)

# Проверка и вывод спецификации агента
print(f'Observation size: {spec.observation_specs[0].shape[0]}')  # Вывести размер входных наблюдений (сколько числовых значений получает агент)
print(f'Continuous action size: {spec.action_spec.continuous_size}')  # Вывести количество непрерывных действий (плавные движения)
print(f'Discrete action branches: {spec.action_spec.discrete_branches}')  # Вывести количество дискретных действий (скачкообразные выборы)

ModuleNotFoundError: No module named 'numpy.typing'

In [ ]:
# Определение кастомной архитектуры нейронной сети для актёра-критика
class CustomActorCriticNet(BaseFeaturesExtractor):  # Наш класс наследует базовый класс для извлечения признаков
    def __init__(self, observation_space, features_dim=128):  # Конструктор: observation_space - размер входов, features_dim - размер выходных признаков
        super(CustomActorCriticNet, self).__init__(observation_space, features_dim)  # Инициализировать родительский класс
        # Первый полносвязный слой: преобразует входные наблюдения в 128 нейронов и переносит на GPU
        self.fc1 = nn.Linear(observation_space.shape[0], 128).to(device)
        # Второй полносвязный слой: преобразует 128 нейронов в 64 нейрона и переносит на GPU
        self.fc2 = nn.Linear(128, 64).to(device)
        # Третий полносвязный слой: преобразует 64 нейрона в итоговые признаки и переносит на GPU
        self.fc3 = nn.Linear(64, features_dim).to(device)
        # Функция активации ReLU (Rectified Linear Unit): превращает отрицательные значения в 0
        self.relu = nn.ReLU()

    # Метод forward выполняется при передаче данных через сеть
    def forward(self, x):  # x - входные наблюдения
        x = self.relu(self.fc1(x))  # Пропустить через первый слой, применить ReLU (выбросить отрицательные значения)
        x = self.relu(self.fc2(x))  # Пропустить через второй слой, применить ReLU
        x = self.fc3(x)  # Пропустить через третий слой (без активации, это выходной слой)
        return x  # Вернуть итоговые признаки

# Словарь с параметрами для настройки политики PPO
policy_kwargs = dict(  # Создать словарь с ключами и значениями
    features_extractor_class=CustomActorCriticNet,  # Использовать нашу кастомную сеть для извлечения признаков
    features_extractor_kwargs=dict(features_dim=128),  # Передать параметр - размер выходных признаков 128
    net_arch=[dict(pi=[64, 32], vf=[64, 32])]  # Архитектура: политика (pi) и функция ценности (vf) имеют по 2 слоя (64 и 32 нейрона)
)

In [ ]:
# Определение класса обёртки для интеграции Unity-среды с Gymnasium API
class UnityGymWrapper(gym.Env):  # Наш класс наследует базовый класс Gymnasium Environment
    def __init__(self, unity_env, behavior_name, spec):  # Конструктор принимает: среду Unity, имя поведения и спецификацию
        super(UnityGymWrapper, self).__init__()  # Инициализировать родительский класс
        self.env = unity_env  # Сохранить ссылку на среду Unity
        self.behavior_name = behavior_name  # Сохранить имя поведения агента
        self.spec = spec  # Сохранить спецификацию агента
        
        # Определить пространство наблюдений (входные данные агента)
        self.observation_space = spaces.Box(  # Box - непрерывное пространство
            low=-np.inf,  # Минимальное значение: минус бесконечность (нет нижней границы)
            high=np.inf,  # Максимальное значение: плюс бесконечность (нет верхней границы)
            shape=(spec.observation_specs[0].shape[0],),  # Форма: кортеж с размером наблюдений
            dtype=np.float32  # Тип данных: 32-битные числа с плавающей точкой
        )
        # Определить пространство действий (выходные данные агента)
        self.action_space = spaces.Box(  # Box - непрерывное пространство
            low=-1.0,  # Минимальное значение действия: -1
            high=1.0,  # Максимальное значение действия: +1
            shape=(spec.action_spec.continuous_size,),  # Форма: кортеж с количеством непрерывных действий
            dtype=np.float32  # Тип данных: 32-битные числа с плавающей точкой
        )

    # Метод для инициализации (начала нового эпизода)
    def reset(self, seed=None, options=None):  # seed и options - опциональные параметры Gymnasium API
        super().reset(seed=seed)  # Вызвать метод родительского класса для установки seed (для воспроизводимости)
        self.env.reset()  # Выполнить сброс Unity-среды
        decision_steps, _ = self.env.get_steps(self.behavior_name)  # Получить наблюдения от агента (decision_steps имеет новые наблюдения)
        obs = decision_steps.obs[0][0]  # Извлечь первое наблюдение агента (первого игрока, первого набора наблюдений)
        info = {}  # Словарь дополнительной информации (пусто)
        return obs, info  # Вернуть наблюдение и информацию

    # Метод для выполнения одного шага в среде
    def step(self, action):  # action - действие, выполняемое агентом
        # Преобразовать действие в правильный формат: 2D массив (1, количество_действий)
        action = np.array(action, dtype=np.float32).reshape(1, -1)  # Преобразовать в массив и изменить форму на (1, N)
        action_tuple = ActionTuple()  # Создать объект для передачи действий
        action_tuple.add_continuous(action)  # Добавить непрерывные действия (только они используются, дискретных нет)

        self.env.set_actions(self.behavior_name, action_tuple)  # Передать действие в Unity-среду
        self.env.step()  # Выполнить один шаг симуляции в Unity

        # Получить результаты после шага
        decision_steps, terminal_steps = self.env.get_steps(self.behavior_name)  # decision_steps - продолжающиеся шаги, terminal_steps - завершённые эпизоды
        done = len(terminal_steps) > 0  # Проверить: если есть завершённые шаги, то эпизод окончен
        if done:  # Если эпизод завершился
            reward = float(terminal_steps.reward[0])  # Получить награду из завершённого шага
            obs = terminal_steps.obs[0][0]  # Получить последнее наблюдение
        else:  # Если эпизод продолжается
            reward = float(decision_steps.reward[0])  # Получить награду из текущего шага
            obs = decision_steps.obs[0][0]  # Получить текущее наблюдение
        info = {}  # Словарь дополнительной информации (пусто)
        truncated = False  # Признак того, что время истекло (не используется, всегда False)

        return obs, reward, done, truncated, info  # Вернуть: наблюдение, награду, завершение, усечение, информацию

    # Метод для закрытия среды
    def close(self):  # Метод вызывается при закрытии окружения
        close_unity_env(self.env)  # Вызвать функцию безопасного закрытия среды Unity

# Создание обёртки Gymnasium для Unity-среды
gym_env = UnityGymWrapper(env, behavior_name, spec)  # Инициализировать обёртку с нашей средой Unity
check_env(gym_env)  # Проверить, что обёртка совместима со стандартом Gymnasium (проверить форматы входов/выходов)

In [ ]:
try:  # Блок try - попытка выполнить обучение
    # Проверить, доступна ли видеокарта NVIDIA с поддержкой CUDA
    device = 'cuda' if torch.cuda.is_available() else 'cpu'  # Если GPU доступна, использовать её; иначе использовать процессор
    print(f"Обучение на устройстве: {device}")  # Вывести информацию о выбранном устройстве

    # Создание и инициализация модели PPO с указанными параметрами
    model = PPO(  # Инициализировать алгоритм PPO (обучение с подкреплением)
        'MlpPolicy',  # Тип политики: многослойный перцептрон (MLP - Multi-Layer Perceptron)
        gym_env,  # Окружение, в котором обучается агент
        policy_kwargs=policy_kwargs,  # Параметры архитектуры нейронной сети (определены выше)
        learning_rate=3e-4,  # Скорость обучения: 0.0003 (как быстро агент учится)
        n_steps=2048,  # Количество шагов перед обновлением весов: 2048 (размер буфера опыта)
        batch_size=64,  # Размер минипакета для обучения: 64 (сколько примеров за раз)
        n_epochs=10,  # Количество эпох обучения: 10 (сколько раз переобработать данные)
        gamma=0.99,  # Коэффициент дисконтирования: 0.99 (вес будущих наград)
        gae_lambda=0.95,  # Параметр GAE (обобщённое адвантажное оценивание): 0.95
        clip_range=0.2,  # Диапазон обрезки: 0.2 (ограничение размера обновлений)
        ent_coef=0.01,  # Коэффициент энтропии: 0.01 (поощрение исследования)
        verbose=1,  # Уровень вывода информации: 1 (показывать информацию о прогрессе)
        device=device  # Устройство для вычисления: GPU или CPU
    )

    # Обучение модели на большом количестве шагов
    model.learn(total_timesteps=100000)  # Обучать агента в течение 100,000 шагов взаимодействия со средой
    model.save('ppo_myagent_gpu')  # Сохранить обученную модель в файл с названием 'ppo_myagent_gpu'

    # Вывод информации о GPU (если используется)
    if device == 'cuda':  # Если используется GPU
        print(f"Используется GPU: {torch.cuda.get_device_name(0)}")  # Вывести название видеокарты
        print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")  # Вывести объём памяти видеокарты в гигабайтах

except Exception as e:  # Если произойдёт ошибка во время обучения
    print(f'Ошибка обучения: {e}')  # Вывести сообщение об ошибке
finally:  # Блок finally - выполняется всегда, даже если была ошибка
    close_unity_env(env)  # Закрыть среду Unity и завершить процессы

In [ ]:
# Импорт необходимых модулей PyTorch
import torch  # Основная библиотека PyTorch для глубокого обучения
from torch.nn import Parameter  # Класс для определения обучаемых параметров сети

# Определение обёртки для экспорта модели в формат ONNX с поддержкой Unity Sentis
class WrapperNet(torch.nn.Module):  # Наш класс наследует базовый класс PyTorch модуля
    def __init__(self, policy, continuous_action_size):  # Конструктор: policy - обученная политика, continuous_action_size - количество действий
        super(WrapperNet, self).__init__()  # Инициализировать родительский класс
        self.policy = policy  # Сохранить обученную политику (нейронную сеть)

        # Определение версии MLAgents (требуется для Sentis)
        # version_number: MLAgents2_0 = 3 (версия 2.0 имеет номер 3)
        version_number = torch.tensor([3], dtype=torch.float32).to(device)  # Создать тензор с номером версии 3 и перенести на GPU
        self.version_number = Parameter(version_number, requires_grad=False)  # Сохранить как необучаемый параметр

        # Определение размера памяти (для рекуррентных сетей)
        # memory_size: 0, так как нет RNN (рекуррентная нейронная сеть не используется)
        memory_size = torch.tensor([0], dtype=torch.float32).to(device)  # Создать тензор с размером памяти 0 и перенести на GPU
        self.memory_size = Parameter(memory_size, requires_grad=False)  # Сохранить как необучаемый параметр

        # Определение формы выходных действий
        # continuous_action_output_shape: [2] для 2 непрерывных действий (например, движение вперёд и поворот)
        continuous_shape = torch.tensor([continuous_action_size], dtype=torch.float32).to(device)  # Создать тензор с размером действий и перенести на GPU
        self.continuous_shape = Parameter(continuous_shape, requires_grad=False)  # Сохранить как необучаемый параметр

    # Метод для выполнения прямого прохода (forward pass) через сеть
    def forward(self, obs, mask):  # obs - наблюдения, mask - маска (для фильтрации недопустимых действий)
        continuous_actions = self.policy(obs, deterministic=True)[0]  # Получить непрерывные действия из политики (deterministic=True означает выбрать лучшее действие)
        continuous_actions = torch.mul(continuous_actions, mask)  # Умножить действия на маску (фиктивное умножение для совместимости с Sentis)
        return continuous_actions, self.continuous_shape, self.version_number, self.memory_size  # Вернуть действия и метаданные

try:  # Блок try - попытка выполнить экспорт в ONNX
    policy = model.policy.to(device)  # Перенести обученную политику на GPU
    continuous_action_size = spec.action_spec.continuous_size  # Получить количество непрерывных действий (2 в нашем случае)
    wrapper_net = WrapperNet(policy, continuous_action_size)  # Создать обёртку с политикой и размером действий
    
    # Создание тестовых входных данных для экспорта (указывают формы входов модели)
    dummy_input = torch.randn(1, spec.observation_specs[0].shape[0]).to(device)  # Случайный тензор размером [1, 6] - 1 агент, 6 наблюдений
    dummy_mask = torch.ones(1, continuous_action_size).to(device)  # Тензор единиц размером [1, 2] - маска для 2 действий
    
    # Экспорт модели в формат ONNX (Open Neural Network Exchange)
    torch.onnx.export(  # Функция для экспорта PyTorch модели в ONNX
        wrapper_net,  # Модель для экспорта
        (dummy_input, dummy_mask),  # Входные данные (используются для определения форм)
        'trained_myagent.onnx',  # Путь и имя выходного файла
        input_names=['obs_0', 'action_masks'],  # Имена входов в ONNX модели
        output_names=['continuous_actions', 'continuous_action_output_shape', 'version_number', 'memory_size'],  # Имена выходов
        dynamic_axes={  # Динамические оси - размеры, которые могут изменяться
            'obs_0': {0: 'batch'},  # Первая ось наблюдений (batch) может изменяться
            'action_masks': {0: 'batch'},  # Первая ось маски (batch) может изменяться
            'continuous_actions': {0: 'batch'},  # Первая ось действий (batch) может изменяться
            'continuous_action_output_shape': {0: 'batch'},  # Первая ось формы (batch) может изменяться
            'version_number': {0: 'batch'},  # Первая ось версии (batch) может изменяться
            'memory_size': {0: 'batch'}  # Первая ось памяти (batch) может изменяться
        },
        opset_version=9,  # Версия ONNX операций: 9 (совместимость с Unity Sentis)
        verbose=False  # Не выводить подробную информацию о процессе экспорта
    )
    print('Модель успешно сохранена: trained_myagent.onnx')  # Вывести сообщение об успехе
    print('Файл существует:', os.path.exists('trained_myagent.onnx'))  # Проверить, что файл действительно создан
except Exception as e:  # Если произойдёт ошибка при экспорте
    print(f'Ошибка экспорта ONNX: {e}')  # Вывести сообщение об ошибке
    print('Попробуйте opset_version=11 или проверьте версию Unity Sentis.')  # Предоставить рекомендации
finally:  # Блок finally - выполняется всегда
    close_unity_env(env)  # Закрыть среду Unity и завершить процессы

============= Diagnostic Run torch.onnx.export version 2.0.1+cu118 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

Модель успешно сохранена: trained_myagent.onnx
Файл существует: True
Ошибка при закрытии среды: No Unity environment is loaded.


In [ ]:
try:  # Блок try - попытка выполнить тестирование
    obs, _ = gym_env.reset()  # Инициализировать среду и получить первое наблюдение (_ - игнорировать информацию)
    for _ in range(1000):  # Цикл: повторить 1000 раз (1000 действий агента)
        action, _ = model.predict(obs, deterministic=True)  # Получить действие от обученной модели (deterministic=True - выбрать лучшее, не случайное)
        obs, reward, done, truncated, info = gym_env.step(action)  # Выполнить действие в среде и получить результаты
        if done or truncated:  # Если эпизод закончился
            print('Эпизод завершён')  # Вывести сообщение
            obs, _ = gym_env.reset()  # Начать новый эпизод и получить первое наблюдение
except Exception as e:  # Если произойдёт ошибка
    print(f'Ошибка тестирования: {e}')  # Вывести сообщение об ошибке
finally:  # Блок finally - выполняется всегда
    close_unity_env(env)  # Закрыть среду Unity и завершить процессы

Ошибка тестирования: No Unity environment is loaded.
Ошибка при закрытии среды: No Unity environment is loaded.
